In [0]:
DEBUG = False
SECTOR = 'all'
FIELD = 'm31'
OBS_FILE = f'$PFS_TARGETING_DATA/data/targeting/m31/M31Catalog_forPFS.csv'
PROJECT_DIR = '/home/enk/pfs'
CONFIG_FILES = [
    f'{PROJECT_DIR}/ga_targeting/configs/netflow/SSP/m31/_common.py',
    f'{PROJECT_DIR}/ga_targeting/configs/netflow/SSP/m31/{FIELD}.py'
]
ASSIGNMENTS_FILE = ''

# Observation time frame in HST
TIME_FROM = '2025-09-13 12:00:00'
TIME_TO = '2025-09-25 12:00:00'

In [0]:
import os, sys
from datetime import datetime
import dateutil.parser
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Ellipse, Circle
from matplotlib.gridspec import GridSpec

import warnings
from erfa import ErfaWarning
warnings.filterwarnings('ignore', category=ErfaWarning, message='.*distance overridden.*')

In [0]:
plt.rc('font', size=6)

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
if DEBUG and 'debug' not in globals():
    import debugpy
    debugpy.listen(('0.0.0.0', 5698))
    debug = True

# Plot definitions

In [0]:
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astroplan import Observer
from astroplan.moon import moon_phase_angle, moon_illumination
import astropy.units as u

from pfs.utils.coordinates import Subaru_POPT2_PFS

from pfs.ga.targeting.targets.m31 import M31_SECTORS #, M31_FIELDS
from pfs.ga.targeting.instrument import *
from pfs.ga.common.diagram import CMD, CCD, FOV, FP, ColorAxis, MagnitudeAxis
from pfs.ga.common.photometry import Photometry, Magnitude, Color
from pfs.ga.targeting.projection import Pointing, WcsProjection
from pfs.ga.targeting.config.netflow import NetflowConfig

In [0]:
field = M31_SECTORS[SECTOR]
hsc = field.get_photometry()
cmd = field.get_cmd()
ccd = field.get_ccd()

In [0]:
pointing = field.get_center()
pointing

In [0]:
wcs = WcsProjection(pointing, proj='TAN')
fov = FOV(projection=wcs)

# Load the config files

In [0]:
config = field.get_netflow_config()
config.load(CONFIG_FILES, ignore_collisions=True)

In [0]:
config.pointings

In [0]:
config.instrument_options

# Load observations

In [0]:
from pfs.ga.targeting.io import ObservationSerializer

In [0]:
obs = field.get_text_observation_reader().read(os.path.expandvars(OBS_FILE))
obs.data.shape

# Calculate the whitening matrix

This basically determines the second moments of the coordinate distribution
and yield the axes of the ellipsoid and the position angle

In [0]:
from pfs.ga.targeting.projection import Whitening
from pfs.ga.targeting.radialprofile import King

In [0]:
# Iteratively determine a whitening matrix that converts the
# spherical coordinates into deprojected coordinates

w = Whitening(projection=wcs)

if FIELD == 'm31':
    w_mask = w.create(obs, iterations=30, s_cut=2.2)
else:
    raise NotImplementedError()

w.M, w.reverse(w.M), field.pos

In [0]:
[dsph_maj, dsph_min] = w.S
dsph_pa = np.degrees(np.angle(w.Vh[0, 0] + 1j * w.Vh[1, 0]))
dsph_maj, dsph_min, dsph_pa

In [0]:
f, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=240)

(wx, wy) = w.apply(obs.get_coords())
ax.plot(wx, wy, '.', ms=0.1)
ax.plot(wx[w_mask], wy[w_mask], '.', ms=0.1)
ax.set_aspect('equal')
ax.grid()

In [0]:
w.M, w.W, w.reverse(w.M)

In [0]:
w.M, w.reverse(w.M), field.pos

# Define pointings

In [0]:
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.coordinates import CartesianRepresentation
from astropy.coordinates.matrix_utilities import rotation_matrix
import astropy.units as u

from pfs.ga.targeting.projection import Pointing

In [0]:
# Cartesian coordinate system
SkyCoord(0, 90, unit='deg').cartesian       # (0, 0, 1)     -> z axis points to north
SkyCoord(0, 0, unit='deg').cartesian         # (1, 0, 0)     -> x axis points to us
SkyCoord(90, 0, unit='deg').cartesian        # (0, 1, 0)     -> y axis points to east

In [0]:
def get_full_rotation(sep, pa, ra, dec):
    r1 = rotation_matrix(sep * u.deg, 'z')  # separation of the pointing center from the center of the object
    r2 = rotation_matrix(pa * u.deg, 'x')   # position angle of the ellipse
    r3 = rotation_matrix(dec * u.deg, 'y')    # declination
    r4 = rotation_matrix(-ra * u.deg, 'z')    # right ascension

    return r4 @ r3 @ r2 @ r1

def calculate_center(sep, pa, ra, dec):
    r = get_full_rotation(sep, pa, ra, dec)
    c = SkyCoord(CartesianRepresentation(1, 0, 0).transform(r))
    return c.ra.value, c.dec.value

# print(r1, r2, r3, r4, r4 @ r3 @ r2 @ r1)

# print(SkyCoord(CartesianRepresentation(1, 0, 0)))
# print(SkyCoord(CartesianRepresentation(1, 0, 0).transform(r1)))
# print(SkyCoord(CartesianRepresentation(1, 0, 0).transform(r2 @ r1)))
# print(SkyCoord(CartesianRepresentation(1, 0, 0).transform(r3 @ r2 @ r1)))
# print(SkyCoord(CartesianRepresentation(1, 0, 0).transform(r4 @ r3 @ r2 @ r1)))

In [0]:
field.pos.ra.value, field.pos.dec.value, calculate_center(1.5, 40, field.pos.ra.value, field.pos.dec.value)

In [0]:
center = field.get_center()
obs_time = Time('2025-09-13T10:00:00')        # midnight in Hawaii

# Use the field defaults
pointings = field.get_pointings(SubaruPFI)

# Use the config file
# pointings = [ p.get_pointing(obs_time=obs_time) for p in config.pointings ]

center, pointings

In [0]:
field

In [0]:
# Generate pointings in a pattern
# Use constants for the center of M31 in decimal degrees

def hex(u,v):
    r = np.sqrt(u**2 + u*v + v**2)
    theta1 = np.rad2deg(np.arctan2(np.sqrt(3)*v, 2*u + v)) % 360
    theta2 = np.rad2deg(np.arctan2(np.sqrt(3)*u, 2*v + u)) % 360
    theta3 = np.rad2deg(np.arctan2(-np.sqrt(3)*v, -2*u - v)) % 360
    theta4 = np.rad2deg(np.arctan2(-np.sqrt(3)*u, -2*v - u)) % 360
    theta = [theta1, theta2, theta3, theta4]
    theta = np.unique(theta)
    theta = np.sort(theta)
    return r, theta

print(hex(4, 1))


In [0]:

SEP = 1.15

decM31 = 41.2687
raM31= 10.6847

decE = decM31 - 0.1*SEP
raE = raM31 + 0.35*SEP/np.cos(np.deg2rad(decE))
paE = -20
fieldpaE = (70 - paE) % 360
decW = decM31 + 0.3*SEP
raW = raM31 - 0.45*SEP/np.cos(np.deg2rad(decW))
paW = -25
fieldpaW = (70 - paW) % 360

raM33 = 23.4583
decM33 = 30.6603
paM33 = -20
fieldpaM33 = 50

pointings = [
    #Pointing(raE, decE, posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),

    #Pointing(*calculate_center(SEP, paE+0, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(SEP, paE+60, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(SEP, paE+120, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(SEP, paE+180, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='diskE_2'),
    Pointing(*calculate_center(SEP, paE+240, raE-0.25*SEP/np.cos(np.deg2rad(decE))*np.sin(np.deg2rad(paE)), decE-0.0*SEP*np.cos(np.deg2rad(paE))), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='diskE_3'),
    Pointing(*calculate_center(SEP, paE+300, raE-0.3*SEP/np.cos(np.deg2rad(decE))*np.sin(np.deg2rad(paE)), decE-0.3*SEP*np.cos(np.deg2rad(paE))), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='GSS_1'),

    #Pointing(*calculate_center(2 * SEP, paE+0, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(2 * SEP, paE+60, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(2 * SEP, paE+120, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(2 * SEP, paE+180, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='haloE_1'),
    Pointing(*calculate_center(2 * SEP, paE+240, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='haloE_3'),
    Pointing(*calculate_center(2 * SEP, paE+300, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='GSS_4'),

    #Pointing(*calculate_center(np.sqrt(3) * SEP, paE+30, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(3) * SEP, paE+90, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(np.sqrt(3) * SEP, paE+150, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='diskE_1'),
    Pointing(*calculate_center(np.sqrt(3) * SEP, paE+210, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='haloE_2'),
    Pointing(*calculate_center(np.sqrt(3) * SEP, paE+270, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='GSS_3'),
    Pointing(*calculate_center(np.sqrt(3) * SEP, paE+330, raE-0.35*SEP/np.cos(np.deg2rad(decE))*np.sin(np.deg2rad(paE)), decE-0.35*SEP*np.cos(np.deg2rad(paE))), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='GSS_2'),

    #Pointing(*calculate_center(3 * SEP, paE+0, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(3 * SEP, paE+60, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(3 * SEP, paE+120, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(3 * SEP, paE+180, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(3 * SEP, paE+240, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='haloE_6'),
    #Pointing(*calculate_center(3 * SEP, paE+300, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),

    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+20, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+40, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+80, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+100, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+140, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+160, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paE+200, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='haloE_4'),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paE+220, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='haloE_5'),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paE+260, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='GSS_5'),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paE+280, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0, label='GSS_6'),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+320, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paE+340, raE, decE), posang=fieldpaE-paE, obs_time=obs_time, stage=0, priority=0),

    Pointing(*calculate_center(SEP, paW+0, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='diskW_4'),
    Pointing(*calculate_center(SEP, paW+60, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='diskW_3'),
    Pointing(*calculate_center(SEP, paW+120, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='diskW_2'),
    #Pointing(*calculate_center(SEP, paW+180, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(SEP, paW+240, raW-0.25*SEP/np.cos(np.deg2rad(decW))*np.sin(np.deg2rad(paW)), decW-0.0*SEP*np.cos(np.deg2rad(paW))), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(SEP, paW+300, raW-0.3*SEP/np.cos(np.deg2rad(decW))*np.sin(np.deg2rad(paW)), decW-0.3*SEP*np.cos(np.deg2rad(paW))), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),

    #Pointing(*calculate_center(2 * SEP, paW+0, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(2 * SEP, paW+60, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_6'),
    Pointing(*calculate_center(2 * SEP, paW+120, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='diskW_1'),
    #Pointing(*calculate_center(2 * SEP, paW+180, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(2 * SEP, paW+240, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(2 * SEP, paW+300, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),

    Pointing(*calculate_center(np.sqrt(3) * SEP, paW+30, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_7'),
    Pointing(*calculate_center(np.sqrt(3) * SEP, paW+90, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_4'),
    #Pointing(*calculate_center(np.sqrt(3) * SEP, paW+150, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(3) * SEP, paW+210, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(3) * SEP, paW+270, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(3) * SEP, paW+330, raW-0.35*SEP/np.cos(np.deg2rad(decW))*np.sin(np.deg2rad(paW)), decW-0.35*SEP*np.cos(np.deg2rad(paW))), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),

    #Pointing(*calculate_center(3 * SEP, paW+0, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(3 * SEP, paW+60, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_9'),
    #Pointing(*calculate_center(3 * SEP, paW+120, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(3 * SEP, paW+180, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(3 * SEP, paW+240, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(3 * SEP, paW+300, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),

    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+20, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paW+40, raW + 0.1*SEP/np.cos(np.deg2rad(decW)), decW + 0.1*SEP), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_10'),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paW+80, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_5'),
    Pointing(*calculate_center(np.sqrt(7) * SEP, paW+100, raW+0.1*SEP/np.cos(np.deg2rad(decW)), decW-0.3*SEP), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_3'),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+140, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+160, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+200, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+220, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+260, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+280, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+320, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    #Pointing(*calculate_center(np.sqrt(7) * SEP, paW+340, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0),
    
    Pointing(*calculate_center(np.sqrt(13) * SEP, paW+46.102, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_12'),
    Pointing(*calculate_center(np.sqrt(13) * SEP, paW+73.898, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_8'),
    Pointing(*calculate_center(np.sqrt(13) * SEP, paW+106.102, raW+0.1*SEP/np.cos(np.deg2rad(decW)), decW-0.4*SEP), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_2'),
    Pointing(*calculate_center(np.sqrt(13) * SEP, paW+106.102, raW+0.2*SEP/np.cos(np.deg2rad(decW)), decW+0.6*SEP), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_1'),

    Pointing(*calculate_center(4 * SEP, paW+60, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='haloW_11'),
    
    Pointing(*calculate_center(np.sqrt(21) * SEP, paW+49.107, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='NWstream_1'),
    Pointing(*calculate_center(5.45 * SEP, paW+52, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='NWstream_2'),
    Pointing(*calculate_center(5 * SEP, paW+60, raW, decW), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='NWstream_3'),
    Pointing(*calculate_center(6 * SEP, paW+60, raW+0.42*SEP/np.cos(np.deg2rad(decW)), decW+0.35*SEP), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='NWstream_4'),
    Pointing(*calculate_center(6 * SEP, paW+60, raW+0.6*SEP/np.cos(np.deg2rad(decW)), decW+1.35*SEP), posang=fieldpaW-paW, obs_time=obs_time, stage=0, priority=0, label='NWstream_5'),
    
    Pointing(*calculate_center(SEP, paE+117, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Nmajoraxis_1'),
    Pointing(*calculate_center(2 * SEP, paE+117, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Nmajoraxis_2'), 
    Pointing(*calculate_center(3* SEP, paE+117, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Nmajoraxis_3'), 
    Pointing(*calculate_center(SEP, paE-16, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Smajoraxis_1'),
    Pointing(*calculate_center(2 * SEP, paE-10, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Smajoraxis_2'), 
    Pointing(*calculate_center(2.75 * SEP, paE-26, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Smajoraxis_3'), 
    Pointing(*calculate_center(1.2 * SEP, paE+47, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Wminoraxis'), 
    Pointing(*calculate_center(0.5 * SEP, paE+227, raE, decE), posang=((fieldpaE-paE)+(fieldpaW-paW))/2, obs_time=obs_time, stage=0, priority=0, label='Eminoraxis'), 
    
    Pointing(*calculate_center(0, paM33+0, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_0'),
    Pointing(*calculate_center(SEP, paM33+0, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_1'),
    Pointing(*calculate_center(SEP, paM33+60, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_2'),
    Pointing(*calculate_center(SEP, paM33+120, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_3'),
    Pointing(*calculate_center(SEP, paM33+180, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_4'),
    Pointing(*calculate_center(SEP, paM33+240, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_5'),
    Pointing(*calculate_center(SEP, paM33+300, raM33, decM33), posang=fieldpaM33, obs_time=obs_time, stage=0, priority=0, label='M33_6'),

]

# Plot the pointings

In [0]:
from astropy.coordinates import SkyCoord as sc
for i, p in enumerate(pointings):
    scp = sc(ra=p.ra, dec=p.dec, unit="deg")
    rastr = scp.ra.to_string(unit='hourangle', sep=':', precision=2, pad=2)
    decstr = scp.dec.to_string(sep=':', precision=1, pad=2, alwayssign=True)
    print(f"        ('sector_0', '{p.label}',   '{rastr}',  '{decstr}', {p.posang}, {p.stage}, 0),")

In [0]:
ra0 = 0 + 42/60 + 44.330/3600
dec0 = 41 + 16/60 + 7.50/3600
ra0 *= 15

nb_catalog_m31 = pd.read_csv('/datascope/subaru/data/targeting/m31/M31_NRGBmap_catalog.csv')
nb_catalog_m31['dec'] = nb_catalog_m31['eta'] + dec0
nb_catalog_m31['ra'] = nb_catalog_m31['xi']/np.cos(nb_catalog_m31['dec']*np.pi/180) + ra0

nb_catalog_m33 = pd.read_csv('/datascope/subaru/data/targeting/m31/M33_PFS-SSP.csv')
nb_catalog_m33 = nb_catalog_m33.rename(columns={'RA': 'ra', 'Dec': 'dec'})

nb_catalog = pd.concat([nb_catalog_m31, nb_catalog_m33], join="outer")

In [0]:
from astropy.io import fits

# MAGNITUDE CUT
mag = 17

sanroman = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-sanroman-2010-gcs.csv')
huxor = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-huxor-2008-gcs.csv')
sarajedini = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-sarajedini-2007-gcs.csv')

# BOLOGNA CATALOG
# Steps:
# 1. Remove NaNs
# 2. Separate candidates and confirmed GCs
# 3. Apply magnitude cut

# Load catalogs (positions and photometry)
bologna_positions = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-bologna-2003-position-classification-gcs.csv', sep=',', decimal='.')
bologna_photometry = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-bologna-2003-gcs-photometry.csv', sep=',', decimal='.', on_bad_lines='skip')

# Remove unnecessary columns from photometry table
bologna_photometry = bologna_photometry.drop(['f_Umag', 'f_Bmag', 'f_Vmag', 'u_Vmag', 'f_Rmag', 'f_Imag', 'Source'], axis=1)

# Replace NaNs with a placeholder value
bologna_photometry = bologna_photometry.fillna(999)

# Rename columns in positions table for consistency
bologna_positions.rename(columns={'RAJ2000': 'RA',
                                  'DEJ2000': 'DEC',
                                  'Com': 'Comment',
                                  'f': 'Global Classification Flag',
                                  'S': 'Spec Confirmed',
                                  'V': 'HighImaging Confirmed'},
                         inplace=True)

# Remove unused columns
del bologna_positions['Source']
del bologna_positions['C']

# Merge the two dataframes using the object identifier
bologna = pd.merge(bologna_positions, bologna_photometry, on='recno', how='outer')
del bologna['M31']  # Unnecessary column

# Keep only valid I magnitudes (999 indicates missing data)
bologna = bologna[bologna['Imag'] != 999]

# Global classification flag:
# [1] - confirmed GC;
# [2] - GC candidates;
# [3] - Controversial object;
# [4] - confirmed galaxy;
# [5] - confirmed H II Region;
# [6] - confirmed star.

print(len(bologna['Global Classification Flag'] == 1))
print(len(bologna['Global Classification Flag'] == 2))

# Separate confirmed GCs and candidates
bologna_confirmed = bologna[bologna['Global Classification Flag'] == 1]
bologna_candidates = bologna[bologna['Global Classification Flag'] == 2]

# Apply magnitude cut
bologna_confirmed_Imag = bologna_confirmed[bologna_confirmed['Imag'] >= mag]
bologna_candidates_Imag = bologna_candidates[bologna_candidates['Imag'] >= mag]

# CHEN CANDIDATES
# Steps:
# 1. Remove NaNs -> no NaNs in table
# 2. Separate candidates and confirmed GCs
# 3. Apply magnitude cut

# Load catalog from Chen et al. 2015
chen = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-chen-2015-newgcs.csv', sep=',', na_values='-', decimal='.')

# Rename coordinate
chen.rename(columns={'RAJ2000': 'RA', 'DEJ2000': 'DEC'}, inplace=True)

# Apply magnitude cut
chen_imag = chen[chen['i'] >= mag]

chen_imag.head()

# WANG CATALOG
# Steps <--- not done! Not able to download complete table from catalog, hub is not working
# 1. Remove NaNs
# 2. Separate candidates and confirmed GCs (if classification data is available)
# 3. Apply magnitude cut

# Load catalog from Wang et al. 2019
wang = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-wang-2019-gcs.csv', sep=',', decimal='.')

# Rename coordinate
wang.rename(columns={'_RA': 'RA', '_DE': 'DEC'}, inplace=True)

# Apply magnitude cut
wang_imag = wang[wang['imag'] >= mag]

print(wang_imag.size)
wang_imag.head()

# SAN ROMAN CATALOG
# Steps:
# 1. Remove NaNs
# 2. Separate candidates and confirmed GCs
# 3. Apply magnitude cut

# Select relevant columns from San Roman catalog
sanroman = sanroman[['MAIN_ID', 'OTYPE_S', 'RA_d', 'DEC_d', 'FILTER_NAME_I', 'FLUX_I',
                     'FLUX_ERROR_I', 'FLUX_SYSTEM_I', 'FILTER_NAME_i', 'FLUX_i',
                     'FLUX_ERROR_i', 'FLUX_SYSTEM_i']]

# Replace missing values with placeholder value
sanroman = sanroman.fillna(999)

# Separate into GC candidates and confirmed GCs based on classification
sanroman_candidates = sanroman[sanroman['OTYPE_S'].isin(['GlobCluster_Candidate', 'Cluster*_Candidate'])]
sanroman_confirmed = sanroman[sanroman['OTYPE_S'].isin(['GlobCluster'])]

# Apply magnitude cut (i-band and I-band)
sanroman_candidates_imag = sanroman_candidates[sanroman_candidates['FLUX_i'] >= mag]
sanroman_candidates_Imag = sanroman_candidates[sanroman_candidates['FLUX_I'] >= mag]
sanroman_confirmed_imag = sanroman_confirmed[sanroman_confirmed['FLUX_i'] >= mag]
sanroman_confirmed_Imag = sanroman_confirmed[sanroman_confirmed['FLUX_I'] >= mag]

# Remove placeholder magnitude rows (valid values only)
sanroman_candidates_imag = sanroman_candidates_imag[sanroman_candidates_imag['FLUX_i'] != 999]
sanroman_candidates_Imag = sanroman_candidates_Imag[sanroman_candidates_Imag['FLUX_I'] != 999]
sanroman_confirmed_imag = sanroman_confirmed_imag[sanroman_confirmed_imag['FLUX_i'] != 999]
sanroman_confirmed_Imag = sanroman_confirmed_Imag[sanroman_confirmed_Imag['FLUX_I'] != 999]

# HUXOR CATALOG
# Steps:
# 1. Remove NaNs
# 2. Separate candidates and confirmed GCs
# 3. Apply magnitude cut

# Select relevant columns from the Huxor catalog
huxor = huxor[['MAIN_ID', 'OTYPE_S', 'RA_d', 'DEC_d', 'FILTER_NAME_I', 'FLUX_I',
               'FLUX_ERROR_I', 'FLUX_SYSTEM_I', 'FILTER_NAME_i', 'FLUX_i',
               'FLUX_ERROR_i', 'FLUX_SYSTEM_i']]

# Replace missing values with placeholder value
huxor = huxor.fillna(999)

# Separate confirmed GCs and candidates based on classification
huxor_confirmed = huxor[huxor['OTYPE_S'] == 'GlobCluster']
huxor_candidates = huxor[huxor['OTYPE_S'] == 'GlobCluster_Candidate']

# Apply magnitude cut (i-band and I-band)
huxor_confirmed_imag = huxor_confirmed[huxor_confirmed['FLUX_i'] >= mag]
huxor_confirmed_Imag = huxor_confirmed[huxor_confirmed['FLUX_I'] >= mag]
huxor_candidates_imag = huxor_candidates[huxor_candidates['FLUX_i'] >= mag]
huxor_candidates_Imag = huxor_candidates[huxor_candidates['FLUX_I'] >= mag]

# Remove placeholder magnitude rows (valid values only)
huxor_confirmed_imag = huxor_confirmed_imag[huxor_confirmed_imag['FLUX_i'] != 999]
huxor_confirmed_Imag = huxor_confirmed_Imag[huxor_confirmed_Imag['FLUX_I'] != 999]
huxor_candidates_imag = huxor_candidates_imag[huxor_candidates_imag['FLUX_i'] != 999]
huxor_candidates_Imag = huxor_candidates_Imag[huxor_candidates_Imag['FLUX_I'] != 999]

# SARAJEDINI CATALOG
# Steps:
# 1. Remove NaNs
# 2. Separate candidates and confirmed GCs
# 3. Apply magnitude cut

# Select relevant columns from Sarajedini catalog
sarajedini = sarajedini[['MAIN_ID', 'OTYPE_S', 'RA_d', 'DEC_d',
                         'FILTER_NAME_I', 'FLUX_I', 'FLUX_ERROR_I', 'FLUX_SYSTEM_I',
                         'FILTER_NAME_i', 'FLUX_i', 'FLUX_ERROR_i', 'FLUX_SYSTEM_i']]

# Replace missing values with placeholder value
sarajedini = sarajedini.fillna(999)

# Separate confirmed GCs and candidates
sarajedini_confirmed = sarajedini[sarajedini['OTYPE_S'] == 'GlobCluster']
sarajedini_candidates = sarajedini[sarajedini['OTYPE_S'] == 'GlobCluster_Candidate']

# Apply magnitude cut (i-band and I-band)
sarajedini_confirmed_imag = sarajedini_confirmed[sarajedini_confirmed['FLUX_i'] >= mag]
sarajedini_confirmed_Imag = sarajedini_confirmed[sarajedini_confirmed['FLUX_I'] >= mag]
sarajedini_candidates_imag = sarajedini_candidates[sarajedini_candidates['FLUX_i'] >= mag]
sarajedini_candidates_Imag = sarajedini_candidates[sarajedini_candidates['FLUX_I'] >= mag]

# Remove placeholder magnitude rows (valid values only)
sarajedini_confirmed_imag = sarajedini_confirmed_imag[sarajedini_confirmed_imag['FLUX_i'] != 999]
sarajedini_confirmed_Imag = sarajedini_confirmed_Imag[sarajedini_confirmed_Imag['FLUX_I'] != 999]
sarajedini_candidates_imag = sarajedini_candidates_imag[sarajedini_candidates_imag['FLUX_i'] != 999]
sarajedini_candidates_Imag = sarajedini_candidates_Imag[sarajedini_candidates_Imag['FLUX_I'] != 999]


# PANDAS CATALOG
# Steps:
# 1. Remove NaNs
# 2. Separate candidates and confirmed GCs
# 3. Apply magnitude cut

# Load FITS table and convert to pandas DataFrame
filename = '/datascope/subaru/data/targeting/m31/globular_clusters/table-pandas-2014-gcs.fits'
hdul = fits.open(filename)[1].data
pandas = pd.DataFrame(hdul)

# Select relevant columns
colunas_desejadas = ['OTYPE_S', 'RA_d', 'DEC_d', 'FLUX_I', 'FLUX_ERROR_I', 'FLUX_SYSTEM_I',
                     'FLUX_i', 'FLUX_ERROR_i', 'FLUX_SYSTEM_i']
pandas = pandas[colunas_desejadas]

# Rename coordinates for consistency
pandas.rename(columns={'RA_d': 'RA', 'DEC_d': 'DEC'}, inplace=True)

# Replace missing values with placeholder value
pandas = pandas.fillna(999)

# Separate confirmed GCs and candidates
pandas_confirmed = pandas[pandas['OTYPE_S'] == 'GlobCluster          ']
pandas_candidates = pandas[pandas['OTYPE_S'] == 'GlobCluster_Candidate']

# Apply magnitude cut (i-band and I-band)
pandas_confirmed_imag = pandas_confirmed[pandas_confirmed['FLUX_i'] >= mag]
pandas_confirmed_Imag = pandas_confirmed[pandas_confirmed['FLUX_I'] >= mag]
pandas_candidates_imag = pandas_candidates[pandas_candidates['FLUX_i'] >= mag]
pandas_candidates_Imag = pandas_candidates[pandas_candidates['FLUX_I'] >= mag]

# Remove placeholder magnitude rows (valid values only)
pandas_confirmed_imag = pandas_confirmed_imag[pandas_confirmed_imag['FLUX_i'] != 400]
pandas_confirmed_Imag = pandas_confirmed_Imag[pandas_confirmed_Imag['FLUX_I'] != 999]
pandas_candidates_imag = pandas_candidates_imag[pandas_candidates_imag['FLUX_i'] != 999]
pandas_candidates_Imag = pandas_candidates_Imag[pandas_candidates_Imag['FLUX_I'] != 999]


# PANDAS WITH CNN CANDIDATES

# Load Pandas CNN candidates catalog
pandasCNN = pd.read_csv('/datascope/subaru/data/targeting/m31/globular_clusters/table-pandas-2015-cnn-candidates.csv', sep=',', decimal='.')  # , on_bad_lines='skip', engine='python'

# Rename coordinate columns for consistency
pandasCNN.rename(columns={'RAJ2000': 'RA', 'DEJ2000': 'DEC'}, inplace=True)

# Replace missing values with placeholder value
pandasCNN = pandasCNN.fillna(999)

pandasCNN.head()





# Combine confirmed GC catalogs

In [0]:
# Combine confirmed GC catalogs into a single table with catalog identifier

# Standardize column names and add catalog identifier
def standardize_gc_catalog(df, catalog_name, ra_col='RA', dec_col='DEC'):
    """Standardize a GC catalog with common columns and add catalog identifier"""
    df = df.copy()
    # Rename coordinate columns if needed
    if ra_col != 'RA':
        df = df.rename(columns={ra_col: 'RA'})
    if dec_col != 'DEC':
        df = df.rename(columns={dec_col: 'DEC'})
    # Add catalog identifier
    df['catalog'] = catalog_name
    # Select only essential columns (RA, DEC, catalog, and any magnitude columns)
    essential_cols = ['RA', 'DEC', 'catalog']
    # Keep magnitude columns if they exist
    mag_cols = [col for col in df.columns if 'FLUX' in col or 'mag' in col or 'Mag' in col]
    return df[essential_cols + mag_cols]

# Standardize each catalog
sarajedini_std = standardize_gc_catalog(sarajedini_confirmed_imag, 'Sarajedini', ra_col='RA_d', dec_col='DEC_d')
pandas_std = standardize_gc_catalog(pandas_confirmed_imag, 'PANDAS')
sanroman_std = standardize_gc_catalog(sanroman_confirmed_imag, 'San Roman', ra_col='RA_d', dec_col='DEC_d')
huxor_std = standardize_gc_catalog(huxor_confirmed_imag, 'Huxor', ra_col='RA_d', dec_col='DEC_d')
bologna_std = standardize_gc_catalog(bologna_confirmed_Imag, 'Bologna')
chen_std = standardize_gc_catalog(chen_imag, 'Chen', ra_col='RAJ2000', dec_col='DECJ2000')
wang_std = standardize_gc_catalog(wang_imag, 'Wang', ra_col='_RA', dec_col='_DE')

# Combine all catalogs
gc_combined = pd.concat([sarajedini_std, pandas_std, sanroman_std, huxor_std, bologna_std, chen_std, wang_std], 
                         ignore_index=True)

print(f"Combined GC catalog: {len(gc_combined)} total objects")
print(f"  Sarajedini: {len(sarajedini_std)}")
print(f"  PANDAS: {len(pandas_std)}")
print(f"  San Roman: {len(sanroman_std)}")
print(f"  Huxor: {len(huxor_std)}")
print(f"  Bologna: {len(bologna_std)}")
print(f"  Chen: {len(chen_std)}")
print(f"  Wang: {len(wang_std)}")

gc_combined.head()

# Filter GCs to only those within hexagonal pointings

In [0]:
# Filter GCs to only those inside the hexagonal PFI pointings
# Similar to the PNe filtering approach

from matplotlib.path import Path

# Create a mask for GCs inside any pointing
gc_mask = np.zeros(len(gc_combined), dtype=bool)

# Loop through each pointing and check which GCs fall inside
for i, pp in enumerate(pointings):
    wfc = SubaruWFC(pp)
    pfi = SubaruPFI(wfc, instrument_options=config.instrument_options)

    # Transform GC coordinates to focal plane coordinates
    gc_fp_complex = pfi.radec_to_fp_pos(
        gc_combined['RA'].values, 
        gc_combined['DEC'].values,
        pp
    )
    
    # Convert complex focal plane positions to x, y coordinates in mm
    gc_fp_x = gc_fp_complex.real
    gc_fp_y = gc_fp_complex.imag
    gc_fp_xy = np.column_stack([gc_fp_x, gc_fp_y])
    
    # Get the hexagonal boundary of the PFI
    bench = pfi._SubaruPFI__bench
    corner_ids = pfi.CORNERS[0]  # Get the outer hexagon corner IDs
    hex_corners_x = [bench.cobras.centers[cid].real for cid in corner_ids]
    hex_corners_y = [bench.cobras.centers[cid].imag for cid in corner_ids]
    hex_boundary = np.column_stack([hex_corners_x, hex_corners_y])
    
    # Create a Path for the hexagon and check which points are inside
    hex_path = Path(hex_boundary)
    gc_in_hex = hex_path.contains_points(gc_fp_xy)
    
    # Update the mask: mark GCs that are within this pointing's hexagonal FOV
    gc_mask |= gc_in_hex

# Filter to only GCs inside pointings
gc_inside = gc_combined[gc_mask].copy()

print(f"Total GCs inside any pointing: {len(gc_inside)}")
print(f"GCs by catalog:")
for catalog_name in gc_inside['catalog'].unique():
    count = len(gc_inside[gc_inside['catalog'] == catalog_name])
    print(f"  {catalog_name}: {count}")

gc_inside.head()

In [0]:
phat = pd.read_csv('/datascope/subaru/data/targeting/m31/HST/phat_brick_corners.txt', skiprows=3, delim_whitespace=True, nrows=23)
phat

In [0]:
cmap = plt.get_cmap('tab20')

f, ax = plt.subplots(1, 1, figsize=(6, 4.7), dpi=240, subplot_kw=dict(projection=wcs.wcs))

# Create a smoothed heat map based on density of stars
from astropy.coordinates import SkyCoord
from scipy.ndimage import gaussian_filter
import astropy.units as u
from PIL import Image

M31_IMAGE_PATH = '/datascope/subaru/data/targeting/m31/M31.jpg'
if M31_IMAGE_PATH and os.path.exists(M31_IMAGE_PATH):
    try:
        from scipy.ndimage import rotate
        img = Image.open(M31_IMAGE_PATH)
        # Convert to numpy array
        img_array = np.array(img)
        # Flip the image horizontally (left to right)
        img_array = np.fliplr(img_array)
        # Rotate the image 110 degrees counterclockwise with white background
        img_array = rotate(img_array, 110, reshape=True, order=1, cval=255)
        
        # Create elliptical alpha mask
        # Ellipse parameters: PA=38 degrees, semi-major axis=1 degree, fade width=0.2 degrees
        h, w = img_array.shape[:2]
        y, x = np.ogrid[:h, :w]
        center_y, center_x = h / 2, w / 2
        
        # Convert pixel coordinates to relative coordinates
        # Approximate: image covers ~10 degrees in RA and ~4 degrees in Dec
        # Image extent is [15+5/60, 5+5/60, 39, 43] = [15.083, 5.083, 39, 43]
        # So width in degrees is ~10 degrees, height is ~4 degrees
        deg_per_pixel_x = 10.0 / w  # degrees per pixel in x (RA direction)
        deg_per_pixel_y = 4.0 / h   # degrees per pixel in y (Dec direction)
        
        # Position relative to center in degrees
        x_deg = (x - center_x) * deg_per_pixel_x
        y_deg = (y - center_y) * deg_per_pixel_y
        
        # Rotate coordinates by position angle (38 degrees)
        pa_rad = np.deg2rad(-38)
        x_rot = x_deg * np.cos(pa_rad) + y_deg * np.sin(pa_rad)
        y_rot = -x_deg * np.sin(pa_rad) + y_deg * np.cos(pa_rad)
        
        # Ellipse semi-major and semi-minor axes (assuming circular for now, adjust if needed)
        a = 1.75  # semi-major axis in degrees
        b = 0.75  # semi-minor axis in degrees (adjust as needed)
        
        # Distance from ellipse center in ellipse-normalized coordinates
        ellipse_dist = np.sqrt((x_rot / a)**2 + (y_rot / b)**2)
        
        # Create alpha mask: opaque inside ellipse, fade rapidly outside
        # Fade width of 0.2 degrees
        fade_width = 0.75  # degrees
        fade_width_normalized = fade_width / a  # normalize by semi-major axis
        
        # Alpha = 1 inside ellipse (ellipse_dist < 1)
        # Alpha fades from 1 to 0 over fade_width beyond ellipse edge
        alpha_mask = np.ones_like(ellipse_dist)
        fade_region = (ellipse_dist >= 1.0) & (ellipse_dist < 1.0 + fade_width_normalized)
        alpha_mask[fade_region] = 1.0 - ((ellipse_dist[fade_region] - 1.0) / fade_width_normalized)
        alpha_mask[ellipse_dist >= 1.0 + fade_width_normalized] = 0.0
        alpha_mask = np.clip(alpha_mask, 0, 1)
        
        # Convert to RGBA
        if img_array.ndim == 2:  # Grayscale
            img_rgba = np.stack([img_array, img_array, img_array, (alpha_mask * 255).astype(np.uint8)], axis=2)
        elif img_array.shape[2] == 3:  # RGB
            img_rgba = np.dstack([img_array, (alpha_mask * 255).astype(np.uint8)])
        else:  # Already has alpha
            img_rgba = img_array.copy()
            img_rgba[:, :, 3] = (alpha_mask * 255).astype(np.uint8)
        
        # Define the extent in sky coordinates (RA, Dec)
        # Adjust these values to match your image's actual coverage
        # Shifted 1 arcminute left (RA + 1/60 degree)
        img_extent = [15 + 40/60, 5 + 40/60, 39 + 14/60, 43 + 14/60]  # [ra_min, ra_max, dec_min, dec_max] in degrees
        ax.imshow(img_rgba, extent=img_extent, transform=ax.get_transform('world'), 
                  zorder=0, origin='lower', aspect='auto')
    except Exception as e:
        print(f"Could not load M31 image: {e}")


M33_IMAGE_PATH = '/datascope/subaru/data/targeting/m31/M33_VST.jpg'
#M33_IMAGE_PATH = '/datascope/subaru/data/targeting/m31/m33_mcmonigal2016.png'
if M33_IMAGE_PATH and os.path.exists(M33_IMAGE_PATH):
    try:
        from scipy.ndimage import rotate
        img = Image.open(M33_IMAGE_PATH)
        # Convert to numpy array
        img_array = np.array(img)
        # Flip the image horizontally (left to right)
        img_array = np.fliplr(img_array)
        # Rotate the image 110 degrees counterclockwise with white background
        #img_array = rotate(img_array, 0, reshape=True, order=1, cval=255)
        
        # Create elliptical alpha mask
        # Ellipse parameters: PA=38 degrees, semi-major axis=1 degree, fade width=0.2 degrees
        h, w = img_array.shape[:2]
        y, x = np.ogrid[:h, :w]
        center_y, center_x = h / 2, w / 2
        
        # Convert pixel coordinates to relative coordinates
        # Approximate: image covers ~10 degrees in RA and ~4 degrees in Dec
        # Image extent is [15+5/60, 5+5/60, 39, 43] = [15.083, 5.083, 39, 43]
        # So width in degrees is ~10 degrees, height is ~4 degrees
        deg_per_pixel_x = 68.1 / 60 / w  # degrees per pixel in x (RA direction)
        deg_per_pixel_y = 57.04 / 60 / h   # degrees per pixel in y (Dec direction)
        
        # Position relative to center in degrees
        x_deg = (x - center_x) * deg_per_pixel_x
        y_deg = (y - center_y) * deg_per_pixel_y
        
        # Rotate coordinates by position angle
        pa_rad = np.deg2rad(-20)
        x_rot = x_deg * np.cos(pa_rad) + y_deg * np.sin(pa_rad)
        y_rot = -x_deg * np.sin(pa_rad) + y_deg * np.cos(pa_rad)
        
        # Ellipse semi-major and semi-minor axes (assuming circular for now, adjust if needed)
        a = 0.5  # semi-major axis in degrees
        b = 0.25  # semi-minor axis in degrees (adjust as needed)
        
        # Distance from ellipse center in ellipse-normalized coordinates
        ellipse_dist = np.sqrt((x_rot / a)**2 + (y_rot / b)**2)
        
        # Create alpha mask: opaque inside ellipse, fade rapidly outside
        # Fade width of 0.2 degrees
        fade_width = 0.75  # degrees
        fade_width_normalized = fade_width / a  # normalize by semi-major axis
        
        # Alpha = 1 inside ellipse (ellipse_dist < 1)
        # Alpha fades from 1 to 0 over fade_width beyond ellipse edge
        alpha_mask = np.ones_like(ellipse_dist)
        fade_region = (ellipse_dist >= 1.0) & (ellipse_dist < 1.0 + fade_width_normalized)
        alpha_mask[fade_region] = 1.0 - ((ellipse_dist[fade_region] - 1.0) / fade_width_normalized)
        alpha_mask[ellipse_dist >= 1.0 + fade_width_normalized] = 0.0
        alpha_mask = np.clip(alpha_mask, 0, 1)
        
        # Convert to RGBA
        if img_array.ndim == 2:  # Grayscale
            img_rgba = np.stack([img_array, img_array, img_array, (alpha_mask * 255).astype(np.uint8)], axis=2)
        elif img_array.shape[2] == 3:  # RGB
            img_rgba = np.dstack([img_array, (alpha_mask * 255).astype(np.uint8)])
        else:  # Already has alpha
            img_rgba = img_array.copy()
            img_rgba[:, :, 3] = (alpha_mask * 255).astype(np.uint8)
        
        # Define the extent in sky coordinates (RA, Dec)
        # Adjust these values to match your image's actual coverage
        # Shifted 1 arcminute left (RA + 1/60 degree)
        img_extent = [15 + 57.04 / 60, 15, 45, 45 + 68.01 / 60]  # [ra_min, ra_max, dec_min, dec_max] in degrees
        #ax.imshow(img_rgba, extent=img_extent, transform=ax.get_transform('world'), 
        #          zorder=0, origin='lower', aspect='auto')
    except Exception as e:
        print(f"Could not load M33 image: {e}")


# Plot individual points FIRST (so they're in the background)
ax.plot(nb_catalog_m31['ra'], nb_catalog_m31['dec'], '.', ms=0.3, color='lightblue', alpha=0.3, transform=ax.get_transform('world'), zorder=1)

# Create 2D histogram
bins_ra = np.linspace(nb_catalog_m31['ra'].min(), nb_catalog_m31['ra'].max(), 300)
bins_dec = np.linspace(nb_catalog_m31['dec'].min(), nb_catalog_m31['dec'].max(), 300)
hist, xedges, yedges = np.histogram2d(nb_catalog_m31['ra'], nb_catalog_m31['dec'], 
                                       bins=[bins_ra, bins_dec])

# Edge-preserving Gaussian smoothing
# Create a mask of where we have data
data_mask = hist > 0

# Apply Gaussian smoothing to both the histogram and the mask
smoothed_hist = gaussian_filter(hist.astype(float), sigma=1.0)
smoothed_mask = gaussian_filter(data_mask.astype(float), sigma=1.0)

# Normalize by the smoothed mask to preserve edge densities
# This prevents density from "leaking" into no-data regions
smoothed_hist = np.divide(smoothed_hist, smoothed_mask, 
                          out=np.zeros_like(smoothed_hist), 
                          where=smoothed_mask > 0.3)

# Mask regions with very low coverage (more aggressive threshold)
smoothed_hist[smoothed_mask < 0.3] = np.nan

# Also mask very low density regions
smoothed_hist[smoothed_hist < 1] = np.nan

# Plot the smoothed density map ON TOP with higher zorder
# Use pcolormesh instead of imshow for proper WCS handling
im = ax.pcolormesh(xedges, yedges, smoothed_hist.T, 
                   cmap='RdBu_r', 
                   transform=ax.get_transform('world'),
                   norm=mpl.colors.LogNorm(vmax=8), alpha=0.8, zorder=2,
                   shading='auto')
cbar = f.colorbar(im, ax=ax, label='NB-selected giant density')
# Remove scientific notation from colorbar - use FuncFormatter directly
cbar.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, pos: f'{int(x)}' if x >= 1 else f'{x:.1f}'))
# Also hide the offset text that appears at the top of the colorbar
cbar.ax.yaxis.offsetText.set_visible(False)
cbar.set_ticks([1, 2, 3, 4, 6])

# Plot the tidal radius from the King profile fit
#ell = king.get_ellipse(R_t)
#fov.plot(ax, ell, native_frame='world', ls='--', lw=1, c='k')
#ell = king.get_ellipse(R_c)
#fov.plot(ax, ell, native_frame='world', ls='-', lw=1, c='k')

# Plot the poiting center
# fov.plot(ax, [field.pos.ra.value], [field.pos.dec.value], fmt='+r', ms=10, native_frame='world')


# Read the PNe catalog
pne_catalog = pd.read_csv('/datascope/subaru/data/targeting/m31/M31_catalogue_for_pfs.dat', delim_whitespace=True, comment='#')

# Filter PNe to only those inside any of the PFI pointings
pne_mask = np.zeros(len(pne_catalog), dtype=bool)

# Import Path for point-in-polygon test
from matplotlib.path import Path

# Plot the pointing footprints with even higher zorder
for i, pp in enumerate(pointings):
    wfc = SubaruWFC(pp)
    pfi = SubaruPFI(wfc, instrument_options=config.instrument_options)

    # Transform PNe coordinates to focal plane coordinates
    pne_fp_complex = pfi.radec_to_fp_pos(
        pne_catalog['RA'].values, 
        pne_catalog['DEC'].values,
        pp
    )
    
    # Convert complex focal plane positions to x, y coordinates in mm
    pne_fp_x = pne_fp_complex.real
    pne_fp_y = pne_fp_complex.imag
    pne_fp_xy = np.column_stack([pne_fp_x, pne_fp_y])
    
    # Get the hexagonal boundary of the PFI
    # The corners define the hexagonal focal plane boundary
    bench = pfi._SubaruPFI__bench
    corner_ids = pfi.CORNERS[0]  # Get the outer hexagon corner IDs
    hex_corners_x = [bench.cobras.centers[cid].real for cid in corner_ids]
    hex_corners_y = [bench.cobras.centers[cid].imag for cid in corner_ids]
    hex_boundary = np.column_stack([hex_corners_x, hex_corners_y])
    
    # Create a Path for the hexagon and check which points are inside
    hex_path = Path(hex_boundary)
    pne_in_hex = hex_path.contains_points(pne_fp_xy)
    
    # Update the mask: mark PNe that are within this pointing's hexagonal FOV
    pne_mask |= pne_in_hex

    pfi.plot_focal_plane(ax, fov, corners=True, c='gray', ls='-', zorder=4)# label=pp.label
    pfi.plot_focal_plane(ax, fov, fill=True, alpha=0.4, c='gray', zorder=4)

# Plot only PNe that are inside at least one pointing
pne_inside = pne_catalog[pne_mask]
ax.plot(pne_inside['RA'], pne_inside['DEC'], 'o', markersize=0.9, c='orange', alpha=0.7, label='planetary nebulae', transform=ax.get_transform('world'), zorder=3)

# Plot GCs by catalog with different markers
gc_markers = {
    'Sarajedini': ('s', 'red', 'Sarajedini GCs'),
    'PANDAS': ('^', 'blue', 'PAndAS GCs'),
    'San Roman': ('d', 'green', 'San Roman GCs'),
    'Huxor': ('v', 'purple', 'Huxor GCs'),
    'Bologna': ('P', 'brown', 'Bologna GCs'),
    'Chen': ('X', 'cyan', 'Chen GCs'),
    'Wang': ('*', 'magenta', 'Wang GCs')
}

gc_label_added = False
for catalog_name, (marker, color, label) in gc_markers.items():
    gc_cat = gc_inside[gc_inside['catalog'] == catalog_name]
    if len(gc_cat) > 0:
        # Only add label to the first GC plot
        plot_label = 'globular clusters' if not gc_label_added else None
        if not gc_label_added:
            gc_label_added = True
        
        ax.plot(gc_cat['RA'], gc_cat['DEC'], '^', markersize=2.5, 
                c='purple', alpha=0.8, label=plot_label, 
                transform=ax.get_transform('world'), zorder=5,
                markeredgecolor='black', markeredgewidth=0.3)

#ax.text(pp.ra, pp.dec, pp.label, color='black', fontsize=8, ha='center', va='center', transform=ax.get_transform('world'), zorder=10)

#ax.plot(raM31, decM31, 'x', ms=8, c='red', label='M31 Center', transform=ax.get_transform('world'), zorder=0)


for i, p in phat.iterrows():
    ra_corners = [p['R.A._1'], p['R.A._2'], p['R.A._3'], p['R.A._4'], p['R.A._1']]
    dec_corners = [p['Decl._1'], p['Decl._2'], p['Decl._3'], p['Decl._4'], p['Decl._1']]
    ax.plot(ra_corners, dec_corners, '-', c='purple', lw=1.5, alpha=0.7, transform=ax.get_transform('world'), zorder=6)
    #ax.text(np.mean(ra_corners), np.mean(dec_corners), f"Brick {int(p['Brick'])}", color='orange', fontsize=6, ha='center', va='center', transform=ax.get_transform('world'), zorder=7)


ax.set_xlim(3.7, -8.5)
ax.set_ylim(-3.5, 3.5)
ax.grid()
ax.set_xlabel('Right Ascension', fontsize=12)
ax.set_ylabel('Declination', fontsize=12)
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

# Create inset for M33 in upper left corner
# M33 coordinates: 01:33:50 +30:39:37 = 23.4583 deg RA, 30.6603 deg Dec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Calculate the same angular scale as the main plot
main_ra_range = 3.7 - (-8.5)  # 12.2 degrees
main_dec_range = 3.5 - (-3.5)  # 7 degrees
main_aspect = main_ra_range / main_dec_range  # ~1.74

# Calculate inset dimensions that match the main plot scale
# Inset size multipliers: width +30%, height -20%
inset_width_multiplier = 1.3
inset_height_multiplier = 0.8

# Base inset size as fraction of main plot
base_width_fraction = 0.2459  # (3.0 / 12.2)
base_height_fraction = 0.4286  # (3.0 / 7.0)

# Calculate actual inset percentages
inset_width_pct = base_width_fraction * 100 * inset_width_multiplier
inset_height_pct = base_height_fraction * 100 * inset_height_multiplier

# Calculate the field of view for the inset to match main plot scale
# Field size = main range * (inset percentage / 100)
inset_ra_range = main_ra_range * (inset_width_pct / 100) * 1.08  # Increased by 8%
inset_dec_range = main_dec_range * (inset_height_pct / 100)

# M33 image size in degrees (57.04 x 68.01 arcmin)
m33_image_width_deg = 68.01 / 60.0  # RA extent in degrees
m33_image_height_deg = 57.04 / 60.0  # Dec extent in degrees
#m33_image_width_deg = 6.0  # RA extent in degrees
#m33_image_height_deg = 6.0  # Dec extent in degrees

# Create inset axes (position in axes coordinates: [left, bottom, width, height])
# Use WCS projection for the inset to match main plot
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
ax_inset = inset_axes(ax, width=f"{inset_width_pct}%", height=f"{inset_height_pct}%", loc='lower right', 
                      bbox_to_anchor=(0.05, 0.05, 0.9, 0.9), 
                      bbox_transform=ax.transAxes,
                      axes_class=plt.matplotlib.projections.get_projection_class('rectilinear'))


# Load and display M33 image
if M33_IMAGE_PATH and os.path.exists(M33_IMAGE_PATH):
    try:
        img_m33 = Image.open(M33_IMAGE_PATH)
        img_m33_array = np.array(img_m33)
        img_m33_array = np.fliplr(img_m33_array)
        
        # Create elliptical mask for M33
        h_m33, w_m33 = img_m33_array.shape[:2]
        y_m33, x_m33 = np.ogrid[:h_m33, :w_m33]
        center_y_m33, center_x_m33 = h_m33 / 2, w_m33 / 2
        
        deg_per_pixel_x_m33 = 68.1 / 60 / w_m33
        deg_per_pixel_y_m33 = 57.04 / 60 / h_m33
        
        x_deg_m33 = (x_m33 - center_x_m33) * deg_per_pixel_x_m33
        y_deg_m33 = (y_m33 - center_y_m33) * deg_per_pixel_y_m33
        
        pa_rad_m33 = np.deg2rad(-70)
        x_rot_m33 = x_deg_m33 * np.cos(pa_rad_m33) + y_deg_m33 * np.sin(pa_rad_m33)
        y_rot_m33 = -x_deg_m33 * np.sin(pa_rad_m33) + y_deg_m33 * np.cos(pa_rad_m33)
        
        a_m33 = 0.5
        b_m33 = 0.25
        
        ellipse_dist_m33 = np.sqrt((x_rot_m33 / a_m33)**2 + (y_rot_m33 / b_m33)**2)
        
        fade_width_m33 = 0.5
        fade_width_normalized_m33 = fade_width_m33 / a_m33
        
        alpha_mask_m33 = np.ones_like(ellipse_dist_m33)
        fade_region_m33 = (ellipse_dist_m33 >= 1.0) & (ellipse_dist_m33 < 1.0 + fade_width_normalized_m33)
        alpha_mask_m33[fade_region_m33] = 1.0 - ((ellipse_dist_m33[fade_region_m33] - 1.0) / fade_width_normalized_m33)
        alpha_mask_m33[ellipse_dist_m33 >= 1.0 + fade_width_normalized_m33] = 0.0
        alpha_mask_m33 = np.clip(alpha_mask_m33, 0, 1)
        
        if img_m33_array.shape[2] == 3:
            img_m33_rgba = np.dstack([img_m33_array, (alpha_mask_m33 * 255).astype(np.uint8)])
        else:
            img_m33_rgba = img_m33_array.copy()
            img_m33_rgba[:, :, 3] = (alpha_mask_m33 * 255).astype(np.uint8)
        
        # Set extent for M33 image - centered on M33, spanning actual image size
        img_extent_m33 = [raM33 + m33_image_width_deg/2, raM33 - m33_image_width_deg/2, 
                          decM33 - m33_image_height_deg/2, decM33 + m33_image_height_deg/2]
        ax_inset.imshow(img_m33_rgba, extent=img_extent_m33, origin='lower', aspect='auto')
        
    except Exception as e:
        print(f"Could not load M33 image in inset: {e}")


# Create WCS projection for M33 inset
m33_pointing = Pointing(raM33, decM33, posang=0, obs_time=obs_time)
m33_wcs = WcsProjection(m33_pointing, proj='TAN')
m33_fov = FOV(projection=m33_wcs)

# Manually configure WCS-like coordinate display
ax_inset.coords = m33_wcs.wcs.wcs

# Plot pointings in the inset
for i, pp in enumerate(pointings):
    wfc_inset = SubaruWFC(pp)
    pfi_inset = SubaruPFI(wfc_inset, instrument_options=config.instrument_options)
    pfi_inset.plot_focal_plane(ax_inset, m33_fov, corners=True, c='gray', ls='-', lw=0.5, zorder=4)
    pfi_inset.plot_focal_plane(ax_inset, m33_fov, fill=True, alpha=0.2, c='gray', zorder=4)

nb_catalog_m33 = nb_catalog_m33[(nb_catalog_m33['gpsf'] <= 25) & (nb_catalog_m33['M31_RGBProb'] >= 0.9)]

# Plot individual points FIRST (so they're in the background)
ax_inset.plot(nb_catalog_m33['ra'], nb_catalog_m33['dec'], '.', ms=0.3, color='lightblue', alpha=0.3, zorder=1)

# Create 2D histogram
bins_ra = np.linspace(nb_catalog_m33['ra'].min(), nb_catalog_m33['ra'].max(), 134)
bins_dec = np.linspace(nb_catalog_m33['dec'].min(), nb_catalog_m33['dec'].max(), 67)
hist, xedges, yedges = np.histogram2d(nb_catalog_m33['ra'], nb_catalog_m33['dec'], 
                                       bins=[bins_ra, bins_dec])

# Edge-preserving Gaussian smoothing
# Create a mask of where we have data
data_mask = hist > 0

# Apply Gaussian smoothing to both the histogram and the mask
smoothed_hist = gaussian_filter(hist.astype(float), sigma=1.0)
smoothed_mask = gaussian_filter(data_mask.astype(float), sigma=1.0)

# Normalize by the smoothed mask to preserve edge densities
# This prevents density from "leaking" into no-data regions
smoothed_hist = np.divide(smoothed_hist, smoothed_mask, 
                          out=np.zeros_like(smoothed_hist), 
                          where=smoothed_mask > 0.3)

# Mask regions with very low coverage (more aggressive threshold)
smoothed_hist[smoothed_mask < 0.3] = np.nan

# Also mask very low density regions
smoothed_hist[smoothed_hist < 1] = np.nan

# Calculate vmax for inset to match the main plot's visual appearance
# Use a percentile to set vmax so the inset has similar color distribution
inset_vmax = np.nanpercentile(smoothed_hist[smoothed_hist > 0], 95)

# Plot the smoothed density map in the inset with adjusted normalization
im_inset = ax_inset.pcolormesh(xedges, yedges, smoothed_hist.T, 
                               cmap='RdBu_r', 
                               norm=mpl.colors.LogNorm(vmax=8), alpha=0.8, zorder=2,
                               shading='auto')

# Configure inset appearance to match main plot
ax_inset.set_xlim(raM33 + inset_ra_range/2, raM33 - inset_ra_range/2)
ax_inset.set_ylim(decM33 - inset_dec_range/2, decM33 + inset_dec_range/2)

# Filter GCs that are within the M33 inset bounds
gc_in_m33_region = gc_inside[
    (gc_inside['RA'] >= ax_inset.get_xlim()[1]) & 
    (gc_inside['RA'] <= ax_inset.get_xlim()[0]) &
    (gc_inside['DEC'] >= ax_inset.get_ylim()[0]) & 
    (gc_inside['DEC'] <= ax_inset.get_ylim()[1])
]

# Plot GCs in the inset (only those within the M33 region)
if len(gc_in_m33_region) > 0:
    ax_inset.plot(gc_in_m33_region['RA'], gc_in_m33_region['DEC'], '^', markersize=1.0, 
        c='purple', alpha=0.8, zorder=5,
        markeredgecolor='black', markeredgewidth=0.3)

# Format axes to match main plot style
from astropy.visualization.wcsaxes import WCSAxes
from astropy import units as u
from astropy.coordinates import Angle

# Set up coordinate formatter for RA (hours:minutes with superscript h and m)
def format_ra_superscript(x, pos):
    angle = Angle(x*u.deg)
    h = int(angle.hms.h)
    m = int(angle.hms.m)
    return rf"{h}$^{{\rm h}}${m:02d}$^{{\rm m}}$"

ax_inset.xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(format_ra_superscript))
# Set RA ticks every 15 minutes (15 minutes = 15/60 hours = 0.25 hours = 3.75 degrees)
ax_inset.xaxis.set_major_locator(plt.matplotlib.ticker.MultipleLocator(3.75))

# Set up coordinate formatter for Dec (integer degrees only)
ax_inset.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda y, pos: f"{int(round(y))}°"
))

# Set grid to show every 2 degrees
ax_inset.yaxis.set_major_locator(plt.matplotlib.ticker.MultipleLocator(2.0))

ax_inset.set_xlabel('', fontsize=6)
ax_inset.set_ylabel('', fontsize=6)
ax_inset.tick_params(labelsize=5)
ax_inset.grid()#True, alpha=0.3, color='white', ls='--', lw=0.5)

ax_inset.set_title('M33', fontsize=7)

# Calculate and plot the elevation and rotator angle

In [0]:
obs = Observer.at_site("Subaru", timezone="US/Hawaii")
hawaii_tz = TimeDelta(-10 * u.hr)

In [0]:
config.pointings

In [0]:
time_from = Time(dateutil.parser.parse(TIME_FROM)) - hawaii_tz
time_to = Time(dateutil.parser.parse(TIME_TO)) - hawaii_tz

time_from, time_to

In [0]:
date_from = np.floor(time_from.jd - 0.5)
date_to = np.ceil(time_to.jd + 0.5)

dates = Time(np.linspace(date_from, date_to + 1, int(date_to - date_from + 2)), format='jd')
dates.to_datetime()

In [0]:
sunrise = obs.sun_rise_time(dates, which='previous') + hawaii_tz
sunset = obs.sun_set_time(dates, which='next') + hawaii_tz

# HST
sunrise.to_datetime(), sunset.to_datetime()

In [0]:
def radecpa_to_altazinr(pos, pa, obs_time):
    altaz = pos.transform_to(AltAz(obstime=obs_time, location=Subaru_POPT2_PFS.Lsbr))
    az = altaz.az.deg
    alt = altaz.alt.deg

    # Instrument rotator angle
    subaru = Subaru_POPT2_PFS.Subaru()
    paa = subaru.radec2inr(pos.ra, pos.dec, obs_time)
    inr = (paa + pa + 180) % 360. - 180.
    # inr = paa + pa

    return alt, az, inr

In [0]:
obs_time = Time(np.linspace(date_from, date_to, 300), format='jd')
obs_time[0], obs_time[-1]

In [0]:
obs_time[0], obs_time[-1]

In [0]:
f, ax = plt.subplots(1, 1, figsize=(7, 5), dpi=240)
ax2 = ax.twinx()

# matplotlib like python datetime
xx = (obs_time + hawaii_tz).to_datetime()

# Shade areas in black when the Sun is below the horizon
for t1, t2 in zip(sunset[:-2].to_datetime(), sunrise[2:].to_datetime()):
    ax.axvspan(t1, t2, color='black', alpha=1)

# Plot the rotator angle for each pointing
for p in pointings:
    #p = p.get_pointing()
    alt, az, inr = radecpa_to_altazinr(p.pos, p.posang, obs_time=(obs_time))
    # alt, az, inr = radecpa_to_altazinr(p.pos, 0, obs_time=(obs_time))

    inr_mask = np.abs(inr) > 178
    inr[inr_mask] = np.nan

    # Instrument rotator angle
    ax2.plot(xx, inr, c='orange')

    # Plot the azimuth angle
    # target_ha = obs.target_hour_angle(obs_time, target=p.pos)
    ax2.plot(xx, ((az + 180.) % 360) - 180., c='green')

# Plot the elevation of the Moon
moon = get_body('Moon', obs_time, Subaru_POPT2_PFS.Lsbr)
moon_phase = moon_illumination(obs_time)
altaz = moon.transform_to(AltAz(obstime=obs_time, location=Subaru_POPT2_PFS.Lsbr))
ax.plot(xx, altaz.alt.degree, '--y')

# Plot target elevation
p = field.get_center_pointing(include_motion=False)
alt, az, inr = radecpa_to_altazinr(p.pos, 0, obs_time=(obs_time))
ax.plot(xx, alt, '-c')

ax.set_xlim((time_from + hawaii_tz).to_datetime(), (time_to + hawaii_tz).to_datetime())

ax.set_yticks(np.linspace(0, 90, 7))
ax.set_ylim(0, 90)

ax.tick_params(axis='x', labelrotation=90)

ax.set_xlabel('time [HST]')
ax.set_ylabel('elevation')
ax.set_title(f'Field: {FIELD}, moon phase: {moon_phase.mean():0.2f}')

ax.grid(True)

ax2.set_yticks(np.linspace(-180, 180, 7))
ax2.set_ylim(-180, 180)

ax2.set_ylabel('rotator angle')

In [0]:
# Calculate the precise time of meridian crossing for each pointing

# Sample time at minute resolution
# Time constants are HST, t_fr and t_to are GMT
t_fr = Time('2025-04-01T18:00:00') - hawaii_tz
t_to = Time('2025-04-02T06:00:00') - hawaii_tz
td = TimeDelta(1 * u.min)
t_fr.jd, t_to.jd, td.jd

In [0]:
t = Time(np.arange(t_fr.jd, t_to.jd, td.jd), format='jd')
tt = (t + hawaii_tz).to_datetime()
t.shape

# Meridian crossing for each pointing in HST

In [0]:
for i, p in enumerate(config.pointings):
    p = p.get_pointing()
    alt, az, inr = radecpa_to_altazinr(p.pos, p.posang, obs_time=(t))

    # Wrap around az so it crossed 0 at the meridian
    az = ((az + 180.) % 360) - 180.

    # Find the zero crossing
    m = (-10 <= az) & (az <= 10)
    ip = interp1d(az[m], t[m].jd)
    t_m = ip(0)

    # Print the meridian-crossing in HST for each pointing
    print(i, p.ra, p.dec, p.posang, (Time(t_m, format='jd') + hawaii_tz).iso)

In [0]:
plt.plot(tt, az)
plt.plot(tt[m], az[m])
plt.axhline(0, c='r')
plt.grid()